In [ ]:
# 1. Imports
import torch  #the base library for deep learning
import torch.nn as nn  #for neural network
import torch.nn.functional as F   #for softmax and cross entropy

# 2. Device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.11.0+cu128
Device: cuda


In [ ]:
!pip install -q datasets  #install  dataset from hugging face

In [ ]:
from datasets import load_dataset  #load the dataset tinystories and take 2k stories

dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train[:2000]"
)

print(dataset)
print(dataset[0])

Dataset({
    features: ['text'],
    num_rows: 2000
})
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [ ]:
import re

unique_words = set()
total_words = 0

for story in dataset["text"]:
    words = re.findall(r"\b\w+\b", story.lower())

    total_words += len(words)
    unique_words.update(words)

print("Total words:", total_words)
print("Unique words:", len(unique_words))

Total words: 335586
Unique words: 5839


In [ ]:
import re

total_words = 0
unique_words = set()

for story in dataset["text"]:
    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", story.lower())

    total_words += len(words)
    unique_words.update(words)

print("Total words:", total_words)
print("Unique words:", len(unique_words))

Total words: 331486
Unique words: 5911


train_data = dataset["train"]

In [ ]:
story_1 = dataset["text"][0]

print(story_1)

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


In [ ]:
text = "\n".join(dataset["text"])   #joining the stories and separate them by new line

print("Characters:", len(text))
print(text[:2000])

Characters: 1702919
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree had many leaves 

In [ ]:
!pip install -q tokenizers  #install the tokenizer for seplitting the text

In [ ]:
from tokenizers import Tokenizer  #BPE byte pair encoding using by llm like gpt to split the data into tokens
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [ ]:
tokenizer = Tokenizer(
    BPE(unk_token="[UNK]")   #the words that can not be tokenize set as unknown
)

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(  #special tokens
    vocab_size=5000,
    special_tokens=[
        "[UNK]",    #unknown
        "[PAD]",    #paddings
        "[BOS]",    #begenings of sequence
        "[EOS]"     #end of sequence
    ]
)

tokenizer.train_from_iterator(        #train the tokenizer to creaate vocab of size 5k
    dataset["text"],
    trainer=trainer
)

vocab_size = tokenizer.get_vocab_size()

print("Vocabulary size:", vocab_size)

Vocabulary size: 5000


In [ ]:
#try a example of tokenization :
sample = "Once upon a time there was a little girl"

encoded_sample = tokenizer.encode(sample)

print("Tokens:")
print(encoded_sample.tokens)

print("\nToken IDs:")
print(encoded_sample.ids)

Tokens:
['Once', 'upon', 'a', 'time', 'there', 'was', 'a', 'little', 'girl']

Token IDs:
[212, 231, 48, 185, 183, 95, 48, 181, 222]


In [ ]:
text = "\n".join(dataset["text"])        #tokenizing the data to get tokens and their IDs

encoded = tokenizer.encode(text)

data = torch.tensor(
    encoded.ids,
    dtype=torch.long
)

print("Number of tokens:", len(data))
print(data[:2000])


Number of tokens: 400952
tensor([218, 149,   8,  ...,  48, 458,   8])


In [ ]:
n = int(0.9 * len(data))         #Trainnig and validation data 0.10 0.90

train_data = data[:n]
val_data = data[n:]

print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))

Training tokens: 360856
Validation tokens: 40096


In [ ]:
batch_size = 32  #num of sequences in each training step
block_size = 64  #the maxixmum num of tokens in context

In [ ]:
def get_batch(split):       #to get the shape of target output that precede the input by one position

    data_source = (
        train_data
        if split == "train"
        else val_data
                  )

    ix = torch.randint(
        0,
        len(data_source) - block_size,
        (batch_size,)
                      )

    x = torch.stack([
        data_source[i:i + block_size]
        for i in ix]
                    )

    y = torch.stack([
        data_source[i + 1:i + block_size + 1]
        for i in ix]
                    )

    return x, y


In [ ]:
xb, yb = get_batch("train")   #checking shapes

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)

Input shape: torch.Size([32, 64])
Target shape: torch.Size([32, 64])


In [ ]:
#Once → Token ID = 125  size of embedding vec for each token

n_embd = 256        #Embedding convert the token id into vector with len =128 when the n_embd increase the model per increased the runtime take more time
n_head = 4           #num of attention heads.
dropout = 0.1       #hyper parameter During training, Dropout randomly and temporarily deactivates some neurons to prevent overfitting

In [ ]:

#Start building the attention
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.query = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(
                    block_size,
                    block_size
                )
            )
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)   #calculate the attention
        q = self.query(x)

        # Attention scores
        wei = q @ k.transpose(-2, -1)         #Compare each query with each key   the results is attention score

        # Scaling
        wei = wei * (                          # scalling To prevent the values ​​from becoming too large and causing problems in Softmax.
            k.shape[-1] ** -0.5
        )

        # Causal mask
        wei = wei.masked_fill(           #the model can not see the future for ex when we have the sentece i love ml learning when we want to predict ml we have to prevent the model from seeing learning by using causal mask
            self.tril[:T, :T] == 0,      #Token 1 → sees  , Token 2 → sees 1,2   ,Token 3 → sees 1,2,3  , Token 4 → sees 1,2,3,4
            float("-inf")
        )

        # Softmax
        wei = F.softmax(                   #this converts the normalized score that comes from k and q to probabilities that have sum =1 this named as attention weights
            wei,
            dim=-1
        )

        wei = self.dropout(wei)

        # Values
        v = self.value(x)

        out = wei @ v   #Use attention weights with v so we know how many importance info for each token

        return out

In [ ]:

#We have 4 heads

class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        num_heads,
        head_size
    ):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(
            num_heads * head_size,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        out = torch.cat(                         #gathering all heads results
            [head(x) for head in self.heads],
            dim=-1
        )

        out = self.proj(out) #We let it slide to linear layer

        return self.dropout(out)

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                n_embd,
                4 * n_embd
            ),

            nn.ReLU(),

            nn.Linear(
                4 * n_embd,
                n_embd
            ),

            nn.Dropout(dropout)
        )

    def forward(self, x):

        return self.net(x)

In [ ]:
class Block(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head
    ):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(
            n_head,
            head_size
        )

        self.ffwd = FeedForward(
            n_embd
        )

        self.ln1 = nn.LayerNorm(
            n_embd
        )

        self.ln2 = nn.LayerNorm(
            n_embd
        )

    def forward(self, x):

        x = x + self.sa(
            self.ln1(x)
        )

        x = x + self.ffwd(
            self.ln2(x)
        )

        return x

In [ ]:
class TinyGPT(nn.Module):

    def __init__(self):
        super().__init__()

        # Token Embedding
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Positional Embedding
        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # Transformer Blocks
        self.blocks = nn.Sequential(
            Block(n_embd, n_head),
            Block(n_embd, n_head)
        )

        # Final normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # Output layer
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(
        self,
        idx,
        targets=None
    ):

        B, T = idx.shape

        # Token embeddings
        tok_emb = self.token_embedding_table(idx)

        # Position embeddings
        pos_emb = self.position_embedding_table(
            torch.arange(
                T,
                device=device
            )
        )

        x = tok_emb + pos_emb

        # Transformer
        x = self.blocks(x)

        x = self.ln_f(x)

        # Prediction
        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.reshape(
                B * T,
                C
            )

            targets = targets.reshape(
                B * T
            )

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

In [ ]:
model = TinyGPT().to(device)

num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Number of parameters: {num_params:,}"
)

Number of parameters: 4,159,880


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

In [ ]:
max_iters = 8000

In [ ]:
for step in range(max_iters):

    xb, yb = get_batch("train")

    xb = xb.to(device)
    yb = yb.to(device)

    logits, loss = model(
        xb,
        yb
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()

    optimizer.step()

    if step % 300 == 0:

        print(
            f"Step {step} | "
            f"Loss: {loss.item():.4f}"
        )

Step 0 | Loss: 8.6809
Step 300 | Loss: 4.5219
Step 600 | Loss: 3.8552
Step 900 | Loss: 3.9060
Step 1200 | Loss: 3.4830
Step 1500 | Loss: 3.3739
Step 1800 | Loss: 3.3158
Step 2100 | Loss: 3.1522
Step 2400 | Loss: 3.1034
Step 2700 | Loss: 2.8898
Step 3000 | Loss: 2.7850
Step 3300 | Loss: 2.7432
Step 3600 | Loss: 2.6982
Step 3900 | Loss: 2.8089
Step 4200 | Loss: 2.6802
Step 4500 | Loss: 2.6922
Step 4800 | Loss: 2.6068
Step 5100 | Loss: 2.5413
Step 5400 | Loss: 2.3814
Step 5700 | Loss: 2.4698
Step 6000 | Loss: 2.3658
Step 6300 | Loss: 2.4122
Step 6600 | Loss: 2.3717
Step 6900 | Loss: 2.3165
Step 7200 | Loss: 2.2185
Step 7500 | Loss: 2.1790
Step 7800 | Loss: 2.1825


In [ ]:
@torch.no_grad()
def generate(
    model,
    idx,
    max_new_tokens,
    temperature=1.0
):

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -block_size:]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :]

        # Temperature
        logits = logits / temperature

        probs = F.softmax(
            logits,
            dim=-1
        )

        idx_next = torch.multinomial(
            probs,
            num_samples=1
        )

        idx = torch.cat(
            (idx, idx_next),
            dim=1
        )

    return idx

In [ ]:
prompt = "my boss fired me,the life is dark , I need some help,So all the employees agreed with the boss."
#prompt="One day, a little girl named Lily found a needle in her room"
prompt_ids = tokenizer.encode(
    prompt
).ids

In [ ]:
context = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=device
)

In [ ]:
generated = generate(
    model,
    context,
    max_new_tokens=20,
    temperature=0.8
)

In [ ]:
generated_ids = generated[0].tolist()

output = tokenizer.decode(
    generated_ids
)

print(output)

my boss f ired me , the life is dark , I need some help , So all the em p lo ye es agreed with the boss . From then on , his mom was very important . She said it was time for being so kind and
